In [1]:
#conda create -n vitals python=3.10 -y
#conda activate vitals
#pip install opencv-python mediapipe==0.10.9 "numpy<2" scipy

"""
=============================================================
  실시간 생체신호 모니터 (고성능/안정화 버전) — rPPG + EVM
=============================================================
"""

import cv2
import mediapipe as mp
import numpy as np
from scipy.signal import butter, filtfilt, detrend
from scipy.fft import rfft, rfftfreq
import collections
import time
import sys

# ─────────────────────────────────────────
#  전역 설정
# ─────────────────────────────────────────
TARGET_FPS       = 30
WINDOW_SECONDS   = 12           # 신호 버퍼 길이(초)
WINDOW_SIZE      = TARGET_FPS * WINDOW_SECONDS

HR_LOW, HR_HIGH  = 0.75, 4.0   # 45 ~ 240 BPM (심박수)
BR_LOW, BR_HIGH  = 0.1,  0.5   # 6 ~ 30 회/분 (호흡수)

EVM_LEVELS       = 5
EVM_ROI_SIZE     = 64

GRAPH_LEN        = 250
GRAPH_H          = 55

MIN_FRAMES       = int(WINDOW_SIZE * 0.35)

# 스무딩 가우시안 팩터 (숫자가 작을수록 부드럽고 반응이 느림)
EMA_ALPHA_HR     = 0.15 
EMA_ALPHA_BR     = 0.10


# ─────────────────────────────────────────
#  신호 처리 유틸
# ─────────────────────────────────────────
def butter_bandpass_filter(data: np.ndarray, low: float, high: float, fs: float, order: int = 4) -> np.ndarray:
    nyq = fs / 2.0
    lo  = np.clip(low  / nyq, 1e-4, 0.9999)
    hi  = np.clip(high / nyq, 1e-4, 0.9999)
    if lo >= hi:
        return data
    b, a = butter(order, [lo, hi], btype='band')
    try:
        return filtfilt(b, a, data)
    except Exception:
        return data


def estimate_fps(timestamps: collections.deque) -> float:
    if len(timestamps) < 2:
        return TARGET_FPS
    arr = np.array(timestamps)
    elapsed = arr[-1] - arr[0]
    return len(arr) / elapsed if elapsed > 0 else TARGET_FPS


def peak_frequency(signal: np.ndarray, fps: float, low: float, high: float) -> float:
    n     = len(signal)
    freqs = rfftfreq(n, d=1.0 / fps)
    spec  = np.abs(rfft(signal * np.hanning(n)))

    mask = (freqs >= low) & (freqs <= high)
    if not np.any(mask):
        return 0.0

    peak_hz = freqs[mask][np.argmax(spec[mask])]
    return float(peak_hz)


# ─────────────────────────────────────────
#  rPPG 프로세서 (심박수)
# ─────────────────────────────────────────
class RPPGProcessor:
    def __init__(self):
        self.r_buf  = collections.deque(maxlen=WINDOW_SIZE)
        self.g_buf  = collections.deque(maxlen=WINDOW_SIZE)
        self.b_buf  = collections.deque(maxlen=WINDOW_SIZE)
        self.ts_buf = collections.deque(maxlen=WINDOW_SIZE)
        
        self.bpm_raw    = 0.0
        self.bpm_smooth = 0.0  # 안정화된 BPM
        self.signal_history = collections.deque(maxlen=GRAPH_LEN)

    def update(self, roi_bgr: np.ndarray) -> float:
        if roi_bgr is None or roi_bgr.size == 0:
            return self.bpm_smooth

        b = float(np.mean(roi_bgr[:, :, 0]))
        g = float(np.mean(roi_bgr[:, :, 1]))
        r = float(np.mean(roi_bgr[:, :, 2]))

        self.r_buf.append(r)
        self.g_buf.append(g)
        self.b_buf.append(b)
        self.ts_buf.append(time.time())

        if len(self.g_buf) < MIN_FRAMES:
            return self.bpm_smooth

        self.bpm_raw = self._compute()
        self._apply_smoothing()
        return self.bpm_smooth

    def _compute(self) -> float:
        fps = estimate_fps(self.ts_buf)
        r = np.array(self.r_buf, dtype=np.float64)
        g = np.array(self.g_buf, dtype=np.float64)
        b = np.array(self.b_buf, dtype=np.float64)

        eps = 1e-8
        Xs  = 3 * r - 2 * g
        Ys  = 1.5 * r + g - 1.5 * b

        Xs -= np.mean(Xs); Xs /= (np.std(Xs) + eps)
        Ys -= np.mean(Ys); Ys /= (np.std(Ys) + eps)

        alpha  = np.std(Xs) / (np.std(Ys) + eps)
        chrom  = Xs - alpha * Ys

        # [개선 1] 디트렌딩(Detrending) 적용하여 베이스라인 흔들림 제거
        chrom = detrend(chrom)

        filtered = butter_bandpass_filter(chrom, HR_LOW, HR_HIGH, fps)
        self.signal_history.clear()
        for v in filtered[-GRAPH_LEN:]:
            self.signal_history.append(v)

        hz  = peak_frequency(filtered, fps, HR_LOW, HR_HIGH)
        bpm = hz * 60.0

        if not (40 <= bpm <= 200):
            return self.bpm_raw if self.bpm_raw > 0 else 0.0
        return bpm

    def _apply_smoothing(self):
        # [개선 2] 지수 이동 평균(EMA) 및 튐(Outlier) 억제
        if self.bpm_raw == 0.0:
            return

        if self.bpm_smooth == 0.0:
            self.bpm_smooth = self.bpm_raw
        else:
            diff = abs(self.bpm_raw - self.bpm_smooth)
            # 심박수가 1초 만에 15 BPM 이상 크게 튀면 노이즈(움직임)로 간주해 반영률 최소화
            dynamic_alpha = 0.02 if diff > 15 else EMA_ALPHA_HR
            self.bpm_smooth = (dynamic_alpha * self.bpm_raw) + ((1 - dynamic_alpha) * self.bpm_smooth)

        self.bpm_smooth = round(self.bpm_smooth, 1)


# ─────────────────────────────────────────
#  EVM 프로세서 (호흡수)
# ─────────────────────────────────────────
class EVMBreathingProcessor:
    def __init__(self):
        self.pyramid_buf   = collections.deque(maxlen=WINDOW_SIZE)
        self.ts_buf        = collections.deque(maxlen=WINDOW_SIZE)
        self.signal_buf    = collections.deque(maxlen=WINDOW_SIZE)
        
        self.bpm_raw       = 0.0
        self.bpm_smooth    = 0.0
        self.signal_history = collections.deque(maxlen=GRAPH_LEN)
        self.prev_pyr_bottom = None

    def _gaussian_pyramid_bottom(self, frame_f32: np.ndarray) -> np.ndarray:
        img = frame_f32.copy()
        for _ in range(EVM_LEVELS - 1):
            img = cv2.pyrDown(img)
        return img

    def update(self, face_bgr: np.ndarray) -> float:
        if face_bgr is None or face_bgr.size == 0:
            return self.bpm_smooth

        small = cv2.resize(face_bgr, (EVM_ROI_SIZE, EVM_ROI_SIZE))
        frame_f = small.astype(np.float32) / 255.0
        pyr_bottom = self._gaussian_pyramid_bottom(frame_f)

        self.pyramid_buf.append(pyr_bottom)
        self.ts_buf.append(time.time())

        if self.prev_pyr_bottom is not None:
            # [개선 3] 움직임 스파이크 억제를 위해 루트(sqrt) 적용 후 스케일 조정
            diff_raw = np.abs(pyr_bottom - self.prev_pyr_bottom)
            diff = np.mean(np.sqrt(diff_raw)) 
        else:
            diff = np.mean(pyr_bottom)

        self.prev_pyr_bottom = pyr_bottom.copy()
        self.signal_buf.append(float(diff))

        if len(self.signal_buf) < MIN_FRAMES:
            return self.bpm_smooth

        self.bpm_raw = self._compute()
        self._apply_smoothing()
        return self.bpm_smooth

    def _compute(self) -> float:
        fps = estimate_fps(self.ts_buf)
        raw = np.array(self.signal_buf, dtype=np.float64)
        
        # [개선 1] 디트렌딩 적용
        raw = detrend(raw)
        
        filtered = butter_bandpass_filter(raw, BR_LOW, BR_HIGH, fps)

        self.signal_history.clear()
        for v in filtered[-GRAPH_LEN:]:
            self.signal_history.append(v)

        hz  = peak_frequency(filtered, fps, BR_LOW, BR_HIGH)
        brpm = hz * 60.0

        if not (4 <= brpm <= 35):
            return self.bpm_raw if self.bpm_raw > 0 else 0.0
        return brpm

    def _apply_smoothing(self):
        if self.bpm_raw == 0.0:
            return

        if self.bpm_smooth == 0.0:
            self.bpm_smooth = self.bpm_raw
        else:
            diff = abs(self.bpm_raw - self.bpm_smooth)
            # 호흡수가 너무 갑자기 튀면 노이즈로 간주
            dynamic_alpha = 0.01 if diff > 5 else EMA_ALPHA_BR
            self.bpm_smooth = (dynamic_alpha * self.bpm_raw) + ((1 - dynamic_alpha) * self.bpm_smooth)

        self.bpm_smooth = round(self.bpm_smooth, 1)


# ─────────────────────────────────────────
#  시각화 및 메인 (생략 없이 통합 유지)
# ─────────────────────────────────────────
def draw_panel(canvas: np.ndarray, x: int, y: int, w: int, h: int, alpha: float = 0.55, color=(15, 15, 20)) -> None:
    sub = canvas[y:y+h, x:x+w]
    if sub.size == 0: return
    rect = np.full_like(sub, color)
    cv2.addWeighted(rect, alpha, sub, 1 - alpha, 0, sub)
    canvas[y:y+h, x:x+w] = sub
    cv2.rectangle(canvas, (x, y), (x+w, y+h), (60, 60, 70), 1)


def draw_signal_graph(canvas: np.ndarray, signal: collections.deque, x: int, y: int, w: int, h: int, color: tuple) -> None:
    draw_panel(canvas, x, y, w, h, alpha=0.7, color=(8, 8, 12))
    if len(signal) < 5: return
    arr = np.array(signal, dtype=np.float64)
    mn, mx = arr.min(), arr.max()
    rng = mx - mn
    if rng < 1e-9: return
    pts = []
    n = len(arr)
    for i, v in enumerate(arr):
        px = x + int(i * (w - 2) / max(n - 1, 1)) + 1
        norm = (v - mn) / rng
        py   = y + h - 4 - int(norm * (h - 8))
        pts.append((px, py))
    for i in range(1, len(pts)):
        cv2.line(canvas, pts[i-1], pts[i], color, 1, cv2.LINE_AA)
    if pts:
        cv2.circle(canvas, pts[-1], 3, color, -1, cv2.LINE_AA)


def draw_progress_bar(canvas: np.ndarray, x: int, y: int, w: int, h: int, pct: float, color: tuple) -> None:
    draw_panel(canvas, x, y, w, h, alpha=0.8, color=(10, 10, 15))
    fill_w = int(w * np.clip(pct, 0, 1))
    if fill_w > 0:
        cv2.rectangle(canvas, (x, y), (x + fill_w, y + h), color, -1)


def put_text(canvas, text, x, y, scale=0.65, color=(240, 240, 240), thickness=1, font=cv2.FONT_HERSHEY_SIMPLEX):
    cv2.putText(canvas, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)


def draw_vitals_overlay(canvas: np.ndarray, rppg: RPPGProcessor, evm: EVMBreathingProcessor, fps_actual: float, buf_pct: float, face_detected: bool) -> None:
    H, W = canvas.shape[:2]
    PW, PH = 280, 310
    px, py = W - PW - 10, 10
    draw_panel(canvas, px, py, PW, PH, alpha=0.72, color=(10, 12, 18))

    put_text(canvas, "VITAL SIGNS MONITOR", px + 12, py + 22, scale=0.5, color=(160, 200, 255), thickness=1)
    cv2.line(canvas, (px + 8, py + 28), (px + PW - 8, py + 28), (50, 60, 80), 1)

    hr = rppg.bpm_smooth
    hr_valid = 45 <= hr <= 180
    hr_col   = (80, 255, 140) if hr_valid else (100, 100, 180)
    put_text(canvas, "HEART RATE", px + 12, py + 50, scale=0.42, color=(150, 150, 180))
    if hr > 0:
        put_text(canvas, f"{hr:.0f}", px + 12, py + 85, scale=1.4, color=hr_col, thickness=2)
        put_text(canvas, "BPM", px + 90, py + 85, scale=0.55, color=hr_col)
    else:
        put_text(canvas, "-- BPM", px + 12, py + 82, scale=0.85, color=(80, 80, 100))

    draw_signal_graph(canvas, rppg.signal_history, px + 8, py + 95, PW - 16, GRAPH_H, color=(80, 255, 140))
    cv2.line(canvas, (px + 8, py + 158), (px + PW - 8, py + 158), (40, 50, 65), 1)

    br = evm.bpm_smooth
    br_valid = 6 <= br <= 30
    br_col   = (80, 180, 255) if br_valid else (100, 100, 180)
    put_text(canvas, "BREATHING RATE", px + 12, py + 175, scale=0.42, color=(150, 150, 180))
    if br > 0:
        put_text(canvas, f"{br:.0f}", px + 12, py + 210, scale=1.4, color=br_col, thickness=2)
        put_text(canvas, "/min", px + 90, py + 210, scale=0.55, color=br_col)
    else:
        put_text(canvas, "-- /min", px + 12, py + 207, scale=0.85, color=(80, 80, 100))

    draw_signal_graph(canvas, evm.signal_history, px + 8, py + 220, PW - 16, GRAPH_H, color=(80, 180, 255))
    cv2.line(canvas, (px + 8, py + 283), (px + PW - 8, py + 283), (40, 50, 65), 1)

    put_text(canvas, f"FPS {fps_actual:.1f}   Buffer {buf_pct*100:.0f}%", px + 12, py + 300, scale=0.38, color=(100, 110, 130))
    draw_progress_bar(canvas, px + 8, py + 303, PW - 16, 4, buf_pct, color=(60, 100, 160))


def draw_roi_boxes(canvas: np.ndarray, fh_box, face_box, face_detected: bool) -> None:
    if not face_detected: return
    if fh_box:
        x1, y1, x2, y2 = fh_box
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (80, 255, 140), 1, cv2.LINE_AA)
        put_text(canvas, "rPPG", x1, y1 - 4, scale=0.38, color=(80, 255, 140))
    if face_box:
        x1, y1, x2, y2 = face_box
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (80, 180, 255), 1, cv2.LINE_AA)
        put_text(canvas, "EVM", x1, y1 - 4, scale=0.38, color=(80, 180, 255))


def draw_status_bar(canvas: np.ndarray, face_detected: bool, buf_pct: float) -> None:
    H, W = canvas.shape[:2]
    draw_panel(canvas, 0, H - 28, W, 28, alpha=0.75, color=(8, 8, 14))

    if not face_detected:
        status = "[ NO FACE DETECTED — 얼굴을 카메라 앞에 위치시키세요 ]"
        put_text(canvas, status, 10, H - 10, scale=0.45, color=(80, 80, 140))
    elif buf_pct < 1.0:
        remaining = f"[ 측정 중... {buf_pct*100:.0f}% 완료 — {int((1-buf_pct)*WINDOW_SECONDS)}초 남음 ]"
        put_text(canvas, remaining, 10, H - 10, scale=0.45, color=(140, 180, 100))
    else:
        put_text(canvas, "[ LIVE ] Do not move.", 10, H - 10, scale=0.45, color=(80, 220, 130))
    put_text(canvas, "Press Q to quit", W - 130, H - 10, scale=0.38, color=(80, 80, 100))


def main():
    print("=" * 55)
    print("  실시간 생체신호 모니터 (rPPG + EVM) - 안정화 버젼")
    print("=" * 55)
    
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    rppg = RPPGProcessor()
    evm  = EVMBreathingProcessor()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] 웹캠을 열 수 없습니다.")
        sys.exit(1)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)

    frame_count = 0
    t_start = time.time()
    last_print_time = time.time()
    fps_smooth = TARGET_FPS

    while True:
        ret, frame = cap.read()
        if not ret: continue

        frame = cv2.flip(frame, 1)
        H, W = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)
        face_detected = results.multi_face_landmarks is not None

        fh_box, face_box = None, None

        if face_detected:
            lms = results.multi_face_landmarks[0].landmark
            
            xs = [int(lm.x * W) for lm in lms]
            ys = [int(lm.y * H) for lm in lms]
            fx1, fy1 = max(0, min(xs) - 8), max(0, min(ys) - 8)
            fx2, fy2 = min(W, max(xs) + 8), min(H, max(ys) + 8)
            face_box = (fx1, fy1, fx2, fy2)

            top_pt = lms[10]
            l_brow, r_brow = lms[107], lms[336]
            l_eye_top, r_eye_top = lms[159], lms[386]

            fh_y1 = max(0, int(top_pt.y * H) - 5)
            fh_y2 = int(min(l_eye_top.y, r_eye_top.y) * H) - 5
            fh_x1 = max(0, int(l_brow.x * W) - 5)
            fh_x2 = min(W, int(r_brow.x * W) + 5)

            if fh_y2 > fh_y1 + 8 and fh_x2 > fh_x1 + 8:
                fh_box = (fh_x1, fh_y1, fh_x2, fh_y2)
                rppg.update(frame[fh_y1:fh_y2, fh_x1:fh_x2])

            if fx2 > fx1 + 8 and fy2 > fy1 + 8:
                evm.update(frame[fy1:fy2, fx1:fx2])

        display = frame.copy()
        if face_detected:
            for lm in results.multi_face_landmarks[0].landmark:
                cv2.circle(display, (int(lm.x * W), int(lm.y * H)), 1, (40, 80, 120), -1)

        draw_roi_boxes(display, fh_box, face_box, face_detected)

        frame_count += 1
        current_time = time.time()
        elapsed = current_time - t_start
        if elapsed > 0:
            fps_smooth = 0.9 * fps_smooth + 0.1 * (frame_count / elapsed)

        buf_pct = min(1.0, len(rppg.g_buf) / WINDOW_SIZE)
        draw_vitals_overlay(display, rppg, evm, fps_smooth, buf_pct, face_detected)
        draw_status_bar(display, face_detected, buf_pct)

        cv2.imshow("Vital Signs Monitor rPPG + EVM", display)

        if current_time - last_print_time >= 1.0:
            hr_val = f"{rppg.bpm_smooth:.1f} BPM" if rppg.bpm_smooth > 0 else "안정화 대기..."
            br_val = f"{evm.bpm_smooth:.1f} 회/분" if evm.bpm_smooth > 0 else "안정화 대기..."
            print(f"[실시간 안정화] 심박수: {hr_val:^15} | 호흡수: {br_val:^15}")
            last_print_time = current_time

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27: break

    cap.release()
    cv2.destroyAllWindows()
    face_mesh.close()

if __name__ == "__main__":
    main()

  실시간 생체신호 모니터 (rPPG + EVM) - 안정화 버젼
[실시간 안정화] 심박수:    안정화 대기...    | 호흡수:    안정화 대기...   
[실시간 안정화] 심박수:    안정화 대기...    | 호흡수:    안정화 대기...   
[실시간 안정화] 심박수:    안정화 대기...    | 호흡수:    안정화 대기...   
[실시간 안정화] 심박수:    안정화 대기...    | 호흡수:    안정화 대기...   
[실시간 안정화] 심박수:    68.0 BPM     | 호흡수:    13.6 회/분    
[실시간 안정화] 심박수:    69.0 BPM     | 호흡수:    11.5 회/분    
[실시간 안정화] 심박수:    67.2 BPM     | 호흡수:     9.8 회/분    
[실시간 안정화] 심박수:    66.0 BPM     | 호흡수:     8.5 회/분    
[실시간 안정화] 심박수:    65.4 BPM     | 호흡수:     7.6 회/분    
[실시간 안정화] 심박수:    67.2 BPM     | 호흡수:     6.8 회/분    
[실시간 안정화] 심박수:    68.1 BPM     | 호흡수:    10.1 회/분    
[실시간 안정화] 심박수:    68.1 BPM     | 호흡수:    10.8 회/분    
[실시간 안정화] 심박수:    68.4 BPM     | 호흡수:    10.4 회/분    
[실시간 안정화] 심박수:    85.4 BPM     | 호흡수:    11.3 회/분    
[실시간 안정화] 심박수:    102.5 BPM    | 호흡수:    11.8 회/분    
[실시간 안정화] 심박수:    107.6 BPM    | 호흡수:    11.1 회/분    
